# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [1]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [2]:
#Caçapava do Sul
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\SC_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais baixados online\4204301\4204301.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais enviados pelo cliente (DOX)\Base GIS\Base GIS\Vetoriais\area_abrangencia_V2.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Concórdia\SIG\SESConcordia.gpkg')
coluna_nome_bacias = 'descriptio'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais criados\Domicílio - SES Existente\popdom.xlsx'
crs = "EPSG:31982"

C:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\venv\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


## Funções auxiliares

In [3]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

import geopandas as gpd

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das LINHAS contidas em cada polígono.
    Requer ambos em CRS projetado (unidades em METROS).
    - Filtra geometrias nulas/vazias
    - Valida polígonos (make_valid/buffer(0))
    - Faz overlay com keep_geom_type=False e filtra só linhas
    - Soma por polígono e adiciona coluna em km
    """
    # Cópias e colunas necessárias
    lines = lines_gdf[["geometry"]].copy()
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()

    # CRS: alinhar se necessário
    if lines.crs != polys.crs:
        polys = polys.to_crs(lines.crs)

    # Limpeza: remover nulos/vazios
    lines = lines[lines.geometry.notnull() & ~lines.geometry.is_empty]
    polys = polys[polys.geometry.notnull() & ~polys.geometry.is_empty]

    # Validar polígonos (Shapely 2 -> make_valid; fallback buffer(0))
    try:
        polys["geometry"] = polys.geometry.make_valid()
    except Exception:
        polys["geometry"] = polys.buffer(0)

    # Overlay SEM restringir tipo de geometria
    inter = gpd.overlay(lines, polys, how="intersection", keep_geom_type=False)

    # Ficar só com partes lineares (descarta GeometryCollection/Polígonos/Pontos)
    inter = inter[inter.geom_type.isin(["LineString", "MultiLineString"])].copy()

    if inter.empty:
        # Retorno “vazio” com colunas esperadas
        return gpd.GeoDataFrame(
            {poly_id_col: [], "Extensão de Rede (m)": [], "Extensão de Rede (km)": []}
        )

    # Comprimento em metros (CRS deve estar em metros!)
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = (
        inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"]
        .sum()
        .sort_values(poly_id_col)
    )
    out["Extensão de Rede (km)"] = out["Extensão de Rede (m)"] / 1000.0
    return out

    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [4]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [5]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) |(domicilios['COD_ESPECIE'] == 8) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [6]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [7]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
setores_aps = setores_aps[setores_aps['v0003']>0]
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [8]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [9]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 68902
A população total na APS em 2022 é de 28859


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [10]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

#dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
dompart_setores_filtrado = dompart_setores
#bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]
bacias_setores_filtrado = bacias_setores

In [11]:
#bacias_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\bacias_setores_filtrado.xlsx'))
#dompart_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\dompart_setores_filtrado.xlsx'))

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [12]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [13]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [14]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
descriptio,,
BRF,1258,3093.246752
Centro,15476,35466.992600
Guilherme Reich,589,1380.908710
Natureza,834,1969.322674
Santa Rita,217,681.420561


##### Exportar excel final

In [15]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [16]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps
resultado_dompop['Dom % APS']
display(resultado_dompop)

A população total na APS em 2022 é de 68902
Os domicílios totais na APS em 2022 é de 28859


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
descriptio,,,,,,
BRF,1258,3093.246752,0.068466,0.043591,0.072625,0.044893
Centro,15476,35466.992600,0.842277,0.536263,0.832717,0.514743
Guilherme Reich,589,1380.908710,0.032056,0.020410,0.032422,0.020042
Natureza,834,1969.322674,0.045390,0.028899,0.046237,0.028581
Santa Rita,217,681.420561,0.011810,0.007519,0.015999,0.009890


In [17]:
tem_agr = int(resultado_dompop['Dom % APS'].sum()*dom_aps)
dom_aps
novdom_aps = int(dom_aps*0.9)
posso_tirar = int(tem_agr - novdom_aps)
print(f'O total é {dom_aps}, agr tenho {tem_agr}. 90% seria {novdom_aps}. Posso tirar {posso_tirar}')

O total é 28859, agr tenho 18374. 90% seria 25973. Posso tirar -7599


# Extensão e Área das Bacias

In [18]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\ServPass+EixoLog (para cálculos).gpkg')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Name")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Name")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

C:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\venv\Lib\site-packages\geopandas\geoseries.py:860: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  return self.notna()


descriptio,BRF,Centro,Guilherme Reich,Natureza,Santa Rita
Domicílios,1258.000000,15476.000000,589.000000,834.000000,217.000000
Dom % bacias,0.068466,0.842277,0.032056,0.045390,0.011810
Dom % APS,0.043591,0.536263,0.020410,0.028899,0.007519
População,3093.246752,35466.992600,1380.908710,1969.322674,681.420561
Pop % bacias,0.072625,0.832717,0.032422,0.046237,0.015999
Pop % APS,0.044893,0.514743,0.020042,0.028581,0.009890
Extensão de Rede (m),NaN,NaN,NaN,NaN,NaN
Área (km²),NaN,NaN,NaN,NaN,NaN
